# Week 3 -- Function 3: Bayesian Optimisation (3D)

In [ ]:
# Cell 2: Imports
import numpy as np
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF
from scipy.stats import norm
import warnings
warnings.filterwarnings('ignore')

print('All libraries loaded successfully.')

In [ ]:
# Cell 3: Load data and inspect
# Function 3: Drug discovery -- minimise adverse side effects (framed as maximise negative).
# Y values represent adverse reaction scores; we maximise to find the least harmful compounds.

X = np.load('../data/function_3/initial_inputs.npy')
Y = np.load('../data/function_3/initial_outputs.npy')

print(f'Input  shape : {X.shape}   (n_samples x n_dimensions)')
print(f'Output shape : {Y.shape}  (n_samples,)')
print()

# Sort descending by Y value
X_list = list(X)
Y_list = list(Y)
pairs = sorted(zip(Y_list, X_list), reverse=True)
Y_sorted = [p[0] for p in pairs]
X_sorted = [p[1] for p in pairs]

print('=' * 70)
print('  All observations (sorted descending by Y)')
print('=' * 70)
for i, (y_val, x_val) in enumerate(zip(Y_sorted, X_sorted)):
    marker = '  <-- best' if i == 0 else ''
    print(f'  [{i+1:2d}]  X = [{x_val[0]:.8f}, {x_val[1]:.8f}, {x_val[2]:.8f}]   Y = {y_val:+.6f}{marker}')
print('=' * 70)

best_Y = Y_sorted[0]
best_X = np.array(X_sorted[0])
print(f'\n  Best Y*  = {best_Y:.6f}')
print(f'  Best X*  = [{best_X[0]:.8f}, {best_X[1]:.8f}, {best_X[2]:.8f}]')

In [ ]:
# Cell 4: Fit GP with log-transformed Y

# Log-transform to handle extreme scale differences across observations
# Y values are adverse reaction scores; log(|Y|) preserves ordering in log-space
Y_fit = np.log(np.abs(Y) + 1e-300)

# Fixed RBF kernel (course style -- no ConstantKernel, no optimisation)
kernel = RBF(length_scale=0.1, length_scale_bounds='fixed')
gp = GaussianProcessRegressor(kernel=kernel, alpha=1e-10)
gp.fit(X, Y_fit)

print('=' * 62)
print('  GP Fitting Results')
print('=' * 62)
print(f'  Kernel                   : {gp.kernel_}')
print(f'  Log-marginal-likelihood  : {gp.log_marginal_likelihood_value_:.4f}')
print()

# Sanity check: predict at best known point
pred_mean, pred_std = gp.predict(best_X.reshape(1, -1), return_std=True)
actual_log = np.log(np.abs(best_Y) + 1e-300)
print('  Sanity check at best known X*:')
print(f'    X*                     = [{best_X[0]:.6f}, {best_X[1]:.6f}, {best_X[2]:.6f}]')
print(f'    GP predicted mean      = {pred_mean[0]:.4f}  (log-space)')
print(f'    Actual log(|Y*|)       = {actual_log:.4f}  (log-space)')
print(f'    GP predicted std       = {pred_std[0]:.8f}')
print('=' * 62)

In [ ]:
# Cell 5: UCB acquisition over 20x20x20 grid

# Build 20x20x20 meshgrid over [0, 1]^3
x1 = np.linspace(0, 1, 20)
x2 = np.linspace(0, 1, 20)
x3 = np.linspace(0, 1, 20)
XX1, XX2, XX3 = np.meshgrid(x1, x2, x3)
X_grid = np.column_stack([XX1.ravel(), XX2.ravel(), XX3.ravel()])  # shape (8000, 3)

# GP predictions across the full grid
post_mean, post_std = gp.predict(X_grid, return_std=True)

# UCB acquisition: mean + beta * std
beta = 2.5  #Higher beta = more exploration at early stage (Week 1, sparse data)
acquisition = post_mean + beta * post_std  # shape (8000,)

# Next query = grid point with highest UCB
best_idx = np.argmax(acquisition)
next_x   = X_grid[best_idx]
next_acq = acquisition[best_idx]

# Portal submission string
portal_string = f'{next_x[0]:.6f}-{next_x[1]:.6f}-{next_x[2]:.6f}'

print('=' * 62)
print('  UCB Acquisition  (beta = 2.5, 20x20x20 grid)')
print('=' * 62)
print(f'  Grid size            : {len(X_grid)} points (20x20x20)')
print(f'  Max UCB value        : {next_acq:.4f}')
print(f'  Next query point     : [{next_x[0]:.6f}, {next_x[1]:.6f}, {next_x[2]:.6f}]')
print()
print('  Portal submission string:')
print(f'  >>> {portal_string} <<<')
print('=' * 62)

In [ ]:
# Cell 7: Summary

print('=' * 62)
print('  SUMMARY -- Bayesian Optimisation Results')
print('=' * 62)
print(f'  Function             : 3D Drug Discovery (adverse side effects)')
print(f'  Objective            : Maximise (maximise negative side effects score)')
print(f'  Kernel               : RBF(length_scale=0.1, fixed)')
print(f'  Acquisition function : UCB  (beta = 2.5)')
print(f'  Y transform          : log(|Y| + 1e-300)')
print(f'  Grid search          : 20x20x20 meshgrid (8,000 points)')
print()
print(f'  Current best X*      : [{best_X[0]:.6f}, {best_X[1]:.6f}, {best_X[2]:.6f}]')
print(f'  Current best Y*      : {best_Y:.6f}')
print()
print(f'  Next query point     : [{next_x[0]:.6f}, {next_x[1]:.6f}, {next_x[2]:.6f}]')
print()
print('  Portal submission string:')
print(f'  >>> {portal_string} <<<')
print('=' * 62)

## Week 1 -- Reflection

**Function**: F3 -- Drug Discovery (Minimise Adverse Reactions) (3D)  
**Domain context**: Maximise negative of side effects; lower compound doses = fewer reactions

### Query
- **Method**: GP + UCB (beta=2.5), RBF kernel (length_scale=0.1, fixed), 20×20×20 meshgrid
- **Query type**: Exploration -- first submission, high uncertainty everywhere
- **Input submitted**: [0.421053, 1.000000, 1.000000]
- **Output received**: -0.476229

### What this result taught us
Max compound 2 and 3 (x2=1, x3=1) gave high adverse reactions (y=-0.476). Reducing compound amounts may lower side effects.

### Strategy for Week 2
Try x2 close to 0 and reduce x3; the GP should guide toward lower-adverse-reaction combinations.